# ARM97 Model Output Variable Browser

This notebook visualizes one manually selected ARM97 model output NetCDF file at a time.

- Edit `MODEL_FILE` in the setup cell, or paste a new file path into the widget.
- Surface variables are variables with `time` but no vertical `lev`/`ilev` dimension; they are plotted as time series.
- Profile variables are variables with `time` and `lev` or `ilev`; the time-series view uses a level slider and plots one level at a time, while the heatmap view shows all levels.
- Profile levels are sorted from low numeric level to high numeric level before plotting.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    return Path("/Users/yunlong/Workshop/SCM-UQ-Workflow")


ROOT = Path(os.environ.get("SCM_UQ_WORKFLOW_ROOT", find_repo_root())).resolve()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".local_cache/matplotlib-cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".local_cache"))

import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

# Manually choose the model output file here. The widget below can also reload a different path.
MODEL_FILE = Path(
    os.environ.get(
        "ARM97_MODEL_FILE",
        str(ROOT / "outputs/arm97_baseline_model_output.nc"),
    )
).expanduser().resolve()

OUT_DIR = ROOT / "notebook_outputs" / "arm97_model_output_all_variables"

assert ROOT.exists(), ROOT
assert MODEL_FILE.exists(), MODEL_FILE

print("repo root:", ROOT)
print("model file:", MODEL_FILE)
print("output dir:", OUT_DIR)


## Inspect Variables

In [ ]:
VERTICAL_DIMS = {"lev", "ilev"}


def is_numeric_variable(var):
    return np.issubdtype(np.dtype(var.dtype), np.number)


def classify_variable(name, var):
    dims = tuple(var.dimensions)
    if "time" not in dims or not is_numeric_variable(var):
        return None
    if name in {"time_bnds"}:
        return None
    if any(dim in dims for dim in VERTICAL_DIMS):
        return "profile"
    return "surface"


def variable_catalog(model_file):
    rows = []
    with Dataset(model_file) as ds:
        for name, var in ds.variables.items():
            category = classify_variable(name, var)
            if category is None:
                continue
            rows.append(
                {
                    "variable": name,
                    "category": category,
                    "dimensions": ", ".join(var.dimensions),
                    "shape": " x ".join(str(x) for x in var.shape),
                    "units": getattr(var, "units", ""),
                    "long_name": getattr(var, "long_name", ""),
                }
            )
    if not rows:
        return pd.DataFrame(columns=["variable", "category", "dimensions", "shape", "units", "long_name"])
    return pd.DataFrame(rows).sort_values(["category", "variable"]).reset_index(drop=True)


def load_catalog(model_file):
    global MODEL_FILE, CATALOG, SURFACE_CATALOG, PROFILE_CATALOG
    MODEL_FILE = Path(model_file).expanduser().resolve()
    if not MODEL_FILE.exists():
        raise FileNotFoundError(MODEL_FILE)
    CATALOG = variable_catalog(MODEL_FILE)
    SURFACE_CATALOG = CATALOG[CATALOG["category"] == "surface"].reset_index(drop=True)
    PROFILE_CATALOG = CATALOG[CATALOG["category"] == "profile"].reset_index(drop=True)
    print(f"Loaded catalog from {MODEL_FILE}")
    print(f"surface variables: {len(SURFACE_CATALOG)}")
    print(f"profile variables: {len(PROFILE_CATALOG)}")
    return CATALOG


catalog = load_catalog(MODEL_FILE)
catalog


## Plot Helpers

In [ ]:
def load_model_time_axis(ds):
    time = ds.variables["time"]
    days = np.asarray(time[:], dtype=np.float64)
    dates = np.array(
        num2date(days, time.units, getattr(time, "calendar", "standard"), only_use_cftime_datetimes=False),
        dtype=object,
    )
    return days, dates


def filled(var_or_array):
    return np.asarray(np.ma.asarray(var_or_array, dtype=np.float64).filled(np.nan), dtype=np.float64)


def reduce_to_time_series(values, dims):
    values = filled(values)
    axes = tuple(i for i, dim in enumerate(dims) if dim != "time")
    if axes:
        return np.nanmean(values, axis=axes)
    return values


def profile_level_dim(dims):
    for dim in ("lev", "ilev"):
        if dim in dims:
            return dim
    raise ValueError(f"no lev/ilev dimension in {dims}")


def reduce_to_time_level(values, dims, level_dim):
    values = filled(values)
    time_axis = dims.index("time")
    level_axis = dims.index(level_dim)
    values = np.moveaxis(values, (time_axis, level_axis), (0, 1))
    if values.ndim > 2:
        values = np.nanmean(values, axis=tuple(range(2, values.ndim)))
    return values


def level_values(ds, level_dim):
    if level_dim in ds.variables and np.issubdtype(np.dtype(ds.variables[level_dim].dtype), np.number):
        values = filled(ds.variables[level_dim][:]).squeeze()
        units = getattr(ds.variables[level_dim], "units", "")
        label = f"{level_dim} ({units})" if units else level_dim
        return values, label
    size = len(ds.dimensions[level_dim])
    return np.arange(size, dtype=np.float64), level_dim


def sort_levels_low_to_high(levels, matrix):
    levels = np.asarray(levels, dtype=np.float64)
    order = np.argsort(levels)
    return levels[order], matrix[:, order]


def color_range(values):
    finite = np.asarray(values)[np.isfinite(values)]
    if finite.size == 0:
        return None, None
    lo, hi = np.nanpercentile(finite, [2, 98])
    if np.isclose(lo, hi):
        return None, None
    return float(lo), float(hi)


def variable_title(name, var):
    long_name = getattr(var, "long_name", "")
    units = getattr(var, "units", "")
    text = f"<b>{name}</b>"
    if long_name:
        text += f": {long_name}"
    if units:
        text += f"<br><sup>Units: {units}</sup>"
    return text


def plot_surface_variable(variable_name):
    with Dataset(MODEL_FILE) as ds:
        _, time_dates = load_model_time_axis(ds)
        var = ds.variables[variable_name]
        dims = tuple(var.dimensions)
        units = getattr(var, "units", "")
        series = reduce_to_time_series(var[:], dims)
        fig = go.Figure(
            go.Scatter(
                x=time_dates,
                y=series,
                mode="lines",
                line=dict(color="#1261A6", width=2.0),
                hovertemplate="%{x}<br>value=%{y:.4g}<extra></extra>",
            )
        )
        subtitle = ""
        if any(dim != "time" and len(ds.dimensions[dim]) > 1 for dim in dims):
            subtitle = "<br><sup>Reduced by averaging non-time dimensions.</sup>"
        fig.update_layout(
            title=dict(text=variable_title(variable_name, var) + subtitle, x=0.01, xanchor="left", font=dict(size=22)),
            template="plotly_white",
            height=500,
            width=1120,
            hovermode="x unified",
            margin=dict(l=80, r=40, t=105, b=70),
        )
        fig.update_xaxes(title="Time")
        fig.update_yaxes(title=units or variable_name)
        return fig


def load_profile_matrix(variable_name):
    with Dataset(MODEL_FILE) as ds:
        _, time_dates = load_model_time_axis(ds)
        var = ds.variables[variable_name]
        dims = tuple(var.dimensions)
        level_dim = profile_level_dim(dims)
        matrix = reduce_to_time_level(var[:], dims, level_dim)
        levels, level_label = level_values(ds, level_dim)
        levels, matrix = sort_levels_low_to_high(levels, matrix)
        units = getattr(var, "units", "")
        title = variable_title(variable_name, var)
    return time_dates, levels, matrix, level_label, units, title


def profile_level_options(variable_name):
    _, levels, _, level_label, _, _ = load_profile_matrix(variable_name)
    options = [(f"{level:g}", int(idx)) for idx, level in enumerate(levels)]
    return options, level_label


def plot_profile_time_series(variable_name, level_index=0):
    time_dates, levels, matrix, level_label, units, title = load_profile_matrix(variable_name)
    level_index = int(np.clip(level_index, 0, len(levels) - 1))
    level = levels[level_index]
    series = matrix[:, level_index]

    fig = go.Figure(
        go.Scatter(
            x=time_dates,
            y=series,
            mode="lines",
            line=dict(color="#1261A6", width=2.0),
            hovertemplate=f"%{{x}}<br>{level_label}={level:g}<br>value=%{{y:.4g}}<extra></extra>",
        )
    )
    fig.update_layout(
        title=dict(
            text=title + f"<br><sup>{level_label}={level:g}; levels are sorted from low numeric level to high numeric level.</sup>",
            x=0.01,
            xanchor="left",
            font=dict(size=22),
        ),
        template="plotly_white",
        height=520,
        width=1120,
        hovermode="x unified",
        margin=dict(l=80, r=40, t=110, b=70),
    )
    fig.update_xaxes(title="Time")
    fig.update_yaxes(title=units or variable_name)
    return fig


def plot_profile_heatmap(variable_name):
    time_dates, levels, matrix, level_label, units, title = load_profile_matrix(variable_name)
    cmin, cmax = color_range(matrix)
    fig = go.Figure(
        go.Heatmap(
            x=time_dates,
            y=levels,
            z=matrix.T,
            colorscale="Viridis",
            zmin=cmin,
            zmax=cmax,
            colorbar=dict(title=units or variable_name),
            hovertemplate="%{x}<br>level=%{y}<br>value=%{z:.4g}<extra></extra>",
        )
    )
    fig.update_layout(
        title=dict(
            text=title + "<br><sup>Levels are sorted from low numeric level to high numeric level.</sup>",
            x=0.01,
            xanchor="left",
            font=dict(size=22),
        ),
        template="plotly_white",
        height=620,
        width=1120,
        margin=dict(l=80, r=80, t=115, b=70),
    )
    fig.update_xaxes(title="Time")
    pressure_axis = any(token in level_label.lower() for token in ("hpa", "pa", "pressure"))
    fig.update_yaxes(title=level_label, autorange="reversed" if pressure_axis else True)
    return fig


## Interactive Variable Browser

Paste or edit the model output path, reload the catalog, then use the separate surface/profile controls below.


In [ ]:
model_path = widgets.Text(
    value=str(MODEL_FILE),
    description="Model file",
    layout=widgets.Layout(width="980px"),
)
reload_button = widgets.Button(description="Load file", button_style="primary", icon="refresh")
load_output = widgets.Output()

surface_dropdown = widgets.Dropdown(description="Surface", layout=widgets.Layout(width="760px"))
surface_output = widgets.Output()

profile_dropdown = widgets.Dropdown(description="Profile", layout=widgets.Layout(width="760px"))
profile_plot_type = widgets.ToggleButtons(
    options=[("time series", "series"), ("heatmap", "heatmap")],
    value="series",
    description="Plot",
)
profile_level = widgets.SelectionSlider(
    options=[("loading", 0)],
    description="Level",
    continuous_update=False,
    readout=True,
    layout=widgets.Layout(width="620px"),
    style={"description_width": "45px"},
)
profile_output = widgets.Output()
_updating_profile_levels = False


def dropdown_options(df):
    options = []
    for _, row in df.sort_values("variable").iterrows():
        label = row["variable"]
        if row["long_name"]:
            label += f" | {row['long_name'][:80]}"
        options.append((label, row["variable"]))
    return options


def preferred_value(options, preferred):
    values = [value for _, value in options]
    for name in preferred:
        if name in values:
            return name
    return values[0] if values else None


def refresh_profile_levels(*_):
    global _updating_profile_levels
    if not profile_dropdown.value:
        profile_level.options = []
        return
    _updating_profile_levels = True
    try:
        options, level_label = profile_level_options(profile_dropdown.value)
        profile_level.description = level_label.split()[0]
        profile_level.options = options
        if options:
            # Choose a mid-tropospheric default when hPa-like levels are available; otherwise use the first level.
            labels_as_float = np.array([float(label) for label, _ in options], dtype=float)
            default_idx = int(np.nanargmin(np.abs(labels_as_float - 500.0))) if np.nanmax(labels_as_float) > 100 else 0
            profile_level.value = options[default_idx][1]
    finally:
        _updating_profile_levels = False


def refresh_dropdowns():
    surface_options = dropdown_options(SURFACE_CATALOG)
    profile_options = dropdown_options(PROFILE_CATALOG)
    surface_dropdown.options = surface_options
    profile_dropdown.options = profile_options
    surface_default = preferred_value(surface_options, ["TREFHT", "TS", "PRECT"])
    profile_default = preferred_value(profile_options, ["T", "Q", "U", "V", "OMEGA", "RELHUM"])
    if surface_default is not None:
        surface_dropdown.value = surface_default
    if profile_default is not None:
        profile_dropdown.value = profile_default
    refresh_profile_levels()


def reload_model(_=None):
    with load_output:
        clear_output(wait=True)
        try:
            catalog = load_catalog(model_path.value)
            display(catalog)
            refresh_dropdowns()
        except Exception as exc:
            print(f"Failed to load model file: {exc!r}")


def redraw_surface(_=None):
    with surface_output:
        clear_output(wait=True)
        if not surface_dropdown.value:
            print("No surface variables found.")
            return
        display(plot_surface_variable(surface_dropdown.value))


def redraw_profile(_=None):
    if _updating_profile_levels:
        return
    with profile_output:
        clear_output(wait=True)
        if not profile_dropdown.value:
            print("No profile variables found.")
            return
        if profile_plot_type.value == "heatmap":
            profile_level.layout.display = "none"
            display(plot_profile_heatmap(profile_dropdown.value))
        else:
            profile_level.layout.display = None
            display(plot_profile_time_series(profile_dropdown.value, level_index=profile_level.value or 0))


def on_profile_variable_change(change=None):
    refresh_profile_levels()
    redraw_profile()


reload_button.on_click(reload_model)
surface_dropdown.observe(redraw_surface, names="value")
profile_dropdown.observe(on_profile_variable_change, names="value")
profile_plot_type.observe(redraw_profile, names="value")
profile_level.observe(redraw_profile, names="value")

refresh_dropdowns()
display(widgets.VBox([widgets.HBox([model_path, reload_button]), load_output]))
display(widgets.HTML("<h3>Surface variables: time series</h3>"))
display(widgets.VBox([surface_dropdown, surface_output]))
display(widgets.HTML("<h3>Profile variables: one-level time series or all-level heatmap</h3>"))
display(widgets.VBox([widgets.HBox([profile_dropdown, profile_plot_type]), profile_level, profile_output]))
redraw_surface()
redraw_profile()


## Save Variable Catalog

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
safe_name = "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in MODEL_FILE.stem)
out = OUT_DIR / f"{safe_name}_variable_catalog.csv"
CATALOG.to_csv(out, index=False)
print("wrote", out)


## PDF Report

Run this section after loading the model output and variable catalog. The variable lists control how many variables are included. The optional `REPORT_START_TIME` and `REPORT_END_TIME` settings crop every plot to the requested time window without changing the included variable count.


In [ ]:
from io import BytesIO
import textwrap

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from reportlab.lib import colors
from reportlab.lib.pagesizes import landscape, letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import Image, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle


REPORT_SURFACE_VARS = None
REPORT_PROFILE_VARS = None

# Optional PDF-only time crop. Use strings like "1997-06-25" or "1997-06-25 12:00".
# Leave as None to keep the full model time range.

REPORT_START_TIME = None
REPORT_END_TIME = None

# REPORT_START_TIME = "1997-06-25"
# REPORT_END_TIME = "1997-07-05"


def _available_report_vars(requested, catalog_df):
    available = set(catalog_df["variable"])
    if requested is None:
        return [name for name in catalog_df["variable"]]
    return [name for name in requested if name in available]


def _describe_variable(variable_name):
    row = CATALOG[CATALOG["variable"] == variable_name]
    if row.empty:
        return "", ""
    first = row.iloc[0]
    return str(first.get("long_name", "")), str(first.get("units", ""))


def _time_window_label(start_time=None, end_time=None):
    start = "run start" if start_time is None else str(start_time)
    end = "run end" if end_time is None else str(end_time)
    return f"{start} to {end}"


def _time_window_mask(time_dates, start_time=None, end_time=None):
    stamps = pd.to_datetime([str(item) for item in time_dates])
    mask = np.ones(len(stamps), dtype=bool)
    if start_time is not None:
        mask &= stamps >= pd.Timestamp(start_time)
    if end_time is not None:
        mask &= stamps <= pd.Timestamp(end_time)
    if not mask.any():
        raise ValueError(f"time window has no samples: {_time_window_label(start_time, end_time)}")
    return mask


def _apply_time_window(time_dates, values, start_time=None, end_time=None):
    mask = _time_window_mask(time_dates, start_time=start_time, end_time=end_time)
    return np.asarray(time_dates, dtype=object)[mask], values[mask]


def _fig_to_png(fig):
    image = BytesIO()
    fig.savefig(image, format="png", dpi=180, bbox_inches="tight")
    plt.close(fig)
    image.seek(0)
    return image


def _surface_report_image(variable_name, start_time=None, end_time=None):
    with Dataset(MODEL_FILE) as ds:
        _, time_dates = load_model_time_axis(ds)
        var = ds.variables[variable_name]
        dims = tuple(var.dimensions)
        units = getattr(var, "units", "")
        long_name = getattr(var, "long_name", "")
        series = reduce_to_time_series(var[:], dims)
    time_dates, series = _apply_time_window(time_dates, series, start_time=start_time, end_time=end_time)

    fig, ax = plt.subplots(figsize=(11, 6.5))
    ax.plot(time_dates, series, color="#1261A6", lw=1.8)
    ax.set_title(f"{variable_name}: {long_name}" if long_name else variable_name, loc="left", fontsize=14, fontweight="semibold")
    ax.set_ylabel(units or variable_name)
    ax.set_xlabel("Time")
    ax.grid(True, alpha=0.25)
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    fig.autofmt_xdate(rotation=0)
    fig.subplots_adjust(left=0.08, right=0.98, top=0.90, bottom=0.12)
    return _fig_to_png(fig)


def _profile_heatmap_report_image(variable_name, start_time=None, end_time=None):
    time_dates, levels, matrix, level_label, units, title = load_profile_matrix(variable_name)
    time_dates, matrix = _apply_time_window(time_dates, matrix, start_time=start_time, end_time=end_time)
    long_name, _ = _describe_variable(variable_name)
    cmin, cmax = color_range(matrix)
    pressure_axis = any(token in level_label.lower() for token in ("hpa", "pa", "pressure"))

    fig, ax = plt.subplots(figsize=(11, 6.5))
    mesh = ax.pcolormesh(time_dates, levels, matrix.T, shading="auto", cmap="viridis", vmin=cmin, vmax=cmax)
    cbar = fig.colorbar(mesh, ax=ax, pad=0.015)
    cbar.set_label(units or variable_name)
    ax.set_title(f"{variable_name}: {long_name}" if long_name else variable_name, loc="left", fontsize=14, fontweight="semibold")
    ax.set_ylabel(level_label)
    ax.set_xlabel("Time")
    if pressure_axis:
        ax.set_ylim(float(np.nanmax(levels)), float(np.nanmin(levels)))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.grid(False)
    fig.autofmt_xdate(rotation=0)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.90, bottom=0.12)
    return _fig_to_png(fig)


def build_model_output_pdf_report(
    output_path=None,
    surface_vars=REPORT_SURFACE_VARS,
    profile_vars=REPORT_PROFILE_VARS,
    start_time=REPORT_START_TIME,
    end_time=REPORT_END_TIME,
):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = Path(output_path or OUT_DIR / f"{MODEL_FILE.stem}_model_output_report.pdf")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    selected_surface = _available_report_vars(surface_vars, SURFACE_CATALOG)
    selected_profile = _available_report_vars(profile_vars, PROFILE_CATALOG)

    doc = SimpleDocTemplate(
        str(output_path),
        pagesize=landscape(letter),
        rightMargin=0.35 * inch,
        leftMargin=0.35 * inch,
        topMargin=0.35 * inch,
        bottomMargin=0.35 * inch,
    )
    styles = getSampleStyleSheet()
    title_style = styles["Title"]
    title_style.fontSize = 18
    title_style.leading = 22
    body_style = styles["BodyText"]
    body_style.fontSize = 9
    body_style.leading = 11

    story = []
    story.append(Paragraph("ARM97 Model Output Report", title_style))
    story.append(Spacer(1, 0.08 * inch))
    story.append(Paragraph(f"Model file: {textwrap.shorten(str(MODEL_FILE), width=145)}", body_style))
    story.append(Spacer(1, 0.06 * inch))
    story.append(Paragraph(f"Time window: {_time_window_label(start_time, end_time)}", body_style))
    story.append(Spacer(1, 0.08 * inch))
    story.append(
        Paragraph(
            "Surface variables are shown as time series. Profile variables are shown as time-level heatmaps. Pressure-coordinate heatmaps place low pressure/high altitude at the top and high pressure/near-surface levels at the bottom.",
            body_style,
        )
    )
    story.append(Spacer(1, 0.14 * inch))

    summary_table = [
        ["Category", "Available", "Included", "Variables"],
        ["Surface", str(len(SURFACE_CATALOG)), str(len(selected_surface)), ", ".join(selected_surface)],
        ["Profile", str(len(PROFILE_CATALOG)), str(len(selected_profile)), ", ".join(selected_profile)],
    ]
    table = Table(summary_table, colWidths=[0.9 * inch, 0.8 * inch, 0.8 * inch, 7.0 * inch])
    table.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#E8EEF7")),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("FONTSIZE", (0, 0), (-1, -1), 8),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ("GRID", (0, 0), (-1, -1), 0.25, colors.HexColor("#CBD5E1")),
                ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F8FAFC")]),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
                ("TOPPADDING", (0, 0), (-1, -1), 4),
            ]
        )
    )
    story.append(table)

    for variable_name in selected_surface:
        story.append(PageBreak())
        story.append(Image(_surface_report_image(variable_name, start_time=start_time, end_time=end_time), width=10.2 * inch, height=6.1 * inch))

    for variable_name in selected_profile:
        story.append(PageBreak())
        story.append(Image(_profile_heatmap_report_image(variable_name, start_time=start_time, end_time=end_time), width=10.2 * inch, height=6.1 * inch))

    doc.build(story)
    print(f"wrote {output_path}")
    return output_path


REPORT_PDF = build_model_output_pdf_report()
REPORT_PDF
